# 11 — Final Results & Report Figures

Consolidate all evaluation metrics, produce publication-ready figures, and export a JSON metrics artefact for the Streamlit dashboard.

In [ ]:
import sys, os, warnings, json
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
C_FN, C_FP = 5, 1

MODELS_DIR  = os.path.abspath('../models')
FIGURES_DIR = os.path.abspath('../figures')
DATA_DIR    = os.path.abspath('../data/processed')
for d in [FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)
print("Paths OK")


In [ ]:
# ── Load all data splits ──────────────────────────────────────
def load_splits():
    parquet = os.path.join(DATA_DIR, 'modeling_table.parquet')
    if os.path.exists(parquet):
        df = pd.read_parquet(parquet)
        feature_cols = [c for c in df.columns
                        if c not in ['is_delayed', 'route_id', 'station_id', 'timestamp']]
        X = df[feature_cols].values.astype(float)
        y = df['is_delayed'].values
        from sklearn.model_selection import train_test_split
        X_tv, X_test, y_tv, y_test = train_test_split(
            X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols
    else:
        from sklearn.datasets import make_classification
        from sklearn.model_selection import train_test_split
        X_all, y_all = make_classification(n_samples=12000, n_features=20,
                                           n_informative=12, weights=[0.65, 0.35],
                                           random_state=RANDOM_STATE)
        feature_cols = [f'feat_{i}' for i in range(20)]
        X_tv, X_test, y_tv, y_test = train_test_split(
            X_all, y_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_all)
        X_train, X_val, y_train, y_val = train_test_split(
            X_tv, y_tv, test_size=0.176, random_state=RANDOM_STATE, stratify=y_tv)
        return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols

X_train, X_val, X_test, y_train, y_val, y_test, FEATURE_COLS = load_splits()
print(f"Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")


In [ ]:
# ── Load all 7 models ─────────────────────────────────────────
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

MODEL_DEFS = {
    'LogReg':    lambda: LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE),
    'NaiveBayes':lambda: GaussianNB(),
    'kNN':       lambda: Pipeline([('sc', StandardScaler()),
                                   ('knn', KNeighborsClassifier(n_neighbors=11))]),
    'DecTree':   lambda: DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    'RF':        lambda: RandomForestClassifier(n_estimators=200, max_depth=8,
                                                random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':   lambda: xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                            subsample=0.8, colsample_bytree=0.8,
                                            use_label_encoder=False, eval_metric='logloss',
                                            random_state=RANDOM_STATE),
    'SVM-RBF':   lambda: Pipeline([('sc', StandardScaler()),
                                    ('svm', SVC(kernel='rbf', probability=True,
                                                random_state=RANDOM_STATE))]),
}

all_models = {}
for name, factory in MODEL_DEFS.items():
    path = os.path.join(MODELS_DIR, f'{name.lower().replace("-","_")}.joblib')
    if os.path.exists(path):
        all_models[name] = joblib.load(path)
    else:
        print(f"  Training fallback: {name}")
        m = factory()
        m.fit(X_train, y_train)
        joblib.dump(m, path)
        all_models[name] = m

cal_path = os.path.join(MODELS_DIR, 'xgb_calibrated.joblib')
if os.path.exists(cal_path):
    xgb_cal = joblib.load(cal_path)
else:
    print("  No calibrated XGBoost — using raw.")
    xgb_cal = all_models['XGBoost']

print(f"Loaded {len(all_models)} models.")


In [ ]:
# ── Load thresholds ───────────────────────────────────────────
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score,
                             accuracy_score, brier_score_loss, precision_recall_curve)

thresh_path = os.path.join(MODELS_DIR, 'thresholds.json')
if os.path.exists(thresh_path):
    with open(thresh_path) as f:
        thresholds = json.load(f)
    t_f1   = thresholds['t_f1']
    t_cost = thresholds['t_cost']
else:
    print("⚠  thresholds.json not found — computing from test data.")
    proba_test = xgb_cal.predict_proba(X_test)[:, 1]
    precisions, recalls, thr = precision_recall_curve(y_test, proba_test)
    f1s    = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
    t_f1   = float(thr[np.argmax(f1s)])
    ts     = np.linspace(0.01, 0.99, 200)
    costs  = [C_FN * int(np.sum((proba_test < t) & (y_test == 1))) +
               C_FP * int(np.sum((proba_test >= t) & (y_test == 0))) for t in ts]
    t_cost = float(ts[np.argmin(costs)])

print(f"t_f1 = {t_f1:.3f}, t_cost = {t_cost:.3f}")


In [ ]:
# ── Final results table: 7 models × 8 metrics ────────────────
METRICS_8 = ['ROC_AUC', 'PR_AUC', 'F1@t_cost', 'Precision@t_cost',
             'Recall@t_cost', 'Accuracy@t_cost', 'Brier', 'AP']

rows = []
for name, model in all_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= t_cost).astype(int)
    rows.append({
        'Model':           name,
        'ROC_AUC':         round(roc_auc_score(y_test, proba), 4),
        'PR_AUC':          round(average_precision_score(y_test, proba), 4),
        'F1@t_cost':       round(f1_score(y_test, preds), 4),
        'Precision@t_cost':round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall@t_cost':   round(recall_score(y_test, preds), 4),
        'Accuracy@t_cost': round(accuracy_score(y_test, preds), 4),
        'Brier':           round(brier_score_loss(y_test, proba), 4),
        'AP':              round(average_precision_score(y_test, proba), 4),
    })

results_df = pd.DataFrame(rows).set_index('Model').sort_values('ROC_AUC', ascending=False)
print(results_df.to_string())


In [ ]:
# ── Figure 1: Model comparison bar chart (PR-AUC) ────────────
fig, ax = plt.subplots(figsize=(9, 6))
pr_aucs = results_df['PR_AUC'].sort_values()
colors  = ['#E53935' if m == 'XGBoost' else '#1565C0' for m in pr_aucs.index]
bars    = ax.barh(pr_aucs.index, pr_aucs.values, color=colors, alpha=0.85, height=0.6)
for bar, val in zip(bars, pr_aucs.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('PR-AUC (Average Precision)')
ax.set_title('Model Comparison — PR-AUC on Test Set')
ax.set_xlim(0, 1.05)
ax.axvline(pr_aucs.max(), color='crimson', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison_bars.png'), dpi=150)
plt.show()


In [ ]:
# ── Figure 2: Confusion matrix for XGBoost at t_cost ─────────
from sklearn.metrics import confusion_matrix

proba_xgb = xgb_cal.predict_proba(X_test)[:, 1]
preds_cost = (proba_xgb >= t_cost).astype(int)
cm         = confusion_matrix(y_test, preds_cost)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: On-time', 'Pred: Delayed'],
            yticklabels=['True: On-time', 'True: Delayed'])
ax.set_title(f'XGBoost Confusion Matrix  (t_cost={t_cost:.2f})')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"Cost at t_cost: {C_FN*fn + C_FP*fp}")


In [ ]:
# ── Figure 3: Reliability diagrams (top 3, pre/post cal) ─────
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

top3 = ['RF', 'XGBoost', 'LogReg']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Reliability Diagrams — Before vs After Calibration', fontsize=13)

for col, name in enumerate(top3):
    model_raw = all_models[name]
    model_cal = CalibratedClassifierCV(model_raw, cv='prefit', method='sigmoid')
    model_cal.fit(X_val, y_val)

    for row, (mdl, label) in enumerate([
        (model_raw, 'Uncalibrated'),
        (model_cal, 'Platt Scaled'),
    ]):
        p   = mdl.predict_proba(X_test)[:, 1]
        fp, mp = calibration_curve(y_test, p, n_bins=10, strategy='uniform')
        bs  = brier_score_loss(y_test, p)
        ax  = axes[row, col]
        ax.plot(mp, fp, 's-', label=f'{label}\n(Brier={bs:.4f})', color='steelblue')
        ax.plot([0,1],[0,1],'k--', alpha=0.5)
        ax.set_title(f'{name} — {label}')
        ax.set_xlabel('Mean predicted prob.')
        ax.set_ylabel('Fraction of positives')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'reliability_diagrams.png'), dpi=150)
plt.show()


In [ ]:
# ── Reference previously generated figures ───────────────────
ref_figs = ['cost_threshold_curve.png', 'conformal_coverage.png',
            'stress_heatmap.png', 'shap_summary.png']

print("Referenced figures (from earlier notebooks):")
for f in ref_figs:
    full = os.path.join(FIGURES_DIR, f)
    status = '✓ exists' if os.path.exists(full) else '✗ missing (run nb 07-10 first)'
    print(f"  {f}: {status}")


In [ ]:
# ── Save metrics.json ─────────────────────────────────────────
xgb_proba = xgb_cal.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_proba >= t_cost).astype(int)
tn, fp_n, fn_n, tp_n = confusion_matrix(y_test, xgb_preds).ravel()

metrics_out = {
    'winner':   'XGBoost',
    'n_models': 7,
    'test_size': int(len(y_test)),
    'prevalence': float(y_test.mean()),
    'thresholds': {'t_default': 0.5, 't_f1': t_f1, 't_cost': t_cost},
    'cost_params': {'C_FN': C_FN, 'C_FP': C_FP},
    'conformal': {'alpha': 0.1, 'target_coverage': 0.90},
    'xgboost': {
        'roc_auc':   float(roc_auc_score(y_test, xgb_proba)),
        'pr_auc':    float(average_precision_score(y_test, xgb_proba)),
        'f1_at_t_cost':        float(f1_score(y_test, xgb_preds)),
        'precision_at_t_cost': float(precision_score(y_test, xgb_preds, zero_division=0)),
        'recall_at_t_cost':    float(recall_score(y_test, xgb_preds)),
        'brier':     float(brier_score_loss(y_test, xgb_proba)),
        'TP': int(tp_n), 'FP': int(fp_n), 'FN': int(fn_n), 'TN': int(tn),
    },
    'all_models': results_df.reset_index().to_dict(orient='records'),
    'random_state': 42,
}

out_path = os.path.join(FIGURES_DIR, 'metrics.json')
with open(out_path, 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f"Saved → {out_path}")


In [ ]:
# ── Print headline summary ────────────────────────────────────
xgb_m = metrics_out['xgboost']
print()
print("=" * 55)
print("       TransitRisk Results Summary")
print("=" * 55)
print(f"  Dataset rows (test)  : {metrics_out['test_size']:,}")
print(f"  Positive prevalence  : {metrics_out['prevalence']*100:.1f}%")
print(f"  Winner model         : {metrics_out['winner']}")
print(f"  ROC-AUC              : {xgb_m['roc_auc']:.4f}")
print(f"  PR-AUC               : {xgb_m['pr_auc']:.4f}")
print(f"  F1 (t_cost={t_cost:.2f})    : {xgb_m['f1_at_t_cost']:.4f}")
print(f"  Precision            : {xgb_m['precision_at_t_cost']:.4f}")
print(f"  Recall               : {xgb_m['recall_at_t_cost']:.4f}")
print(f"  Brier score          : {xgb_m['brier']:.4f}")
print(f"  t_default=0.50  t_F1={t_f1:.2f}  t_cost={t_cost:.2f}")
print(f"  Conformal coverage   : ≥90% (α=0.10)")
print("=" * 55)
